# Projeto 03 - Iluminação: Casa Minecraft

Evolução do Projeto 02: o mesmo cenário 3D (**OpenGL** com PyOpenGL e **GLFW**, modelos Wavefront `.obj` com textura),
agora com **iluminação de Phong** calculada no *fragment shader*: componentes **ambiente, difusa e especular**.

### As luzes

| Tecla | Luz | Cor | Onde atua |
|---|---|---|---|
| `1` | **Ambiente** (global) | branca | todo o cenário; com **≤ 25%** o céu vira a skybox de **noite**, com **> 25%** volta a de **dia** |
| `2` | **Bloco de redstone** que o Enderman carrega (anda com ele) | vermelha | **somente ambiente externo** |
| `3` `4` | **Lanternas** do teto (1 e 2) | alaranjada | **somente ambiente interno** |
| `5` `6` | **Tochas de redstone** (1 e 2) | vermelha | **somente ambiente interno** |

### Como separamos o ambiente interno do externo

* Todo `Objeto` tem um `ambiente` (`'interno'` ou `'externo'`) e toda `Luz` também.
* O shader recebe o ambiente do objeto que está desenhando (`ambienteObjeto`) e **só soma a luz** se
  `luz.ambiente == ambienteObjeto` (e se ela estiver ligada).
* A casa e o terreno são **uma malha só** que tem faces dentro e fora da sala. Por isso, ao carregar esses dois modelos,
  cada face é classificada: se um ponto logo à frente dela (na direção da normal) cai dentro da sala, a face é **interna**;
  senão, é **externa**. Cada grupo vira uma "parte" separada, desenhada com o seu ambiente.

### Comandos

`W A S D - Move a câmera (frente / esquerda / trás / direita)`

`Mouse - Direciona a câmera`

`Scroll - Zoom (campo de visão)`

`C - Altera a liberdade da câmera (voo livre / caminhada)`

`↑ ↓ ← → - Move o Enderman (relativo à câmera: ↑ = para longe da câmera). Ele gira para onde anda, não entra na casa e anda por todo o terreno`

`Olhar para os olhos do Enderman - Ele treme (aponte o centro da tela para o rosto dele)`

`1 - Liga/desliga a luz ambiente`

`2 - Liga/desliga a luz do bloco de redstone (externa)`

`3 4 - Liga/desliga a lanterna 1 / 2 (interna)`

`5 6 - Liga/desliga a tocha de redstone 1 / 2 (interna)`

`T G - Aumenta / diminui a luz ambiente`

`Y H - Aumenta / diminui a reflexão difusa (de todos os objetos)`

`U J - Aumenta / diminui a reflexão especular (de todos os objetos)`

`P - Ativa/Desativa a visualização poligonal (malhas)`

`Espaço - Reseta as condições iniciais`

`ESC - Sai`

O estado atual (luzes ligadas, níveis) aparece no **título da janela**.

### Estrutura do notebook

1. Bibliotecas, janela e shaders
2. **Carregamento**: texturas, modelos `.obj`, céu de dia e de noite (tudo vai para a classe `Geometria`)
3. **Cena**: `Objeto` (modelo + posição + material) e `Luz`
4. Envio dos dados para a GPU
5. Matrizes, `Camera`, `Controles` e teclado/mouse
6. Desenho e loop principal

Integrantes:
```
Luís Henrique de Queiroz Veras -- 14592414
Vinícius de Sá Ferreira -- 15491650
```

## 1. Bibliotecas, janela e shaders

### Importando bibliotecas

* **PyOpenGL / glfw**: desenho 3D, janela e eventos de teclado/mouse.
* **numpy**: vértices e matrizes de transformação 4x4 (as matrizes de objeto são feitas à mão, como no Projeto 01).
* **PyGLM**: usado apenas na câmera (`lookAt`, `perspective`) e nos vetores da câmera.
* **PIL**: leitura das imagens de textura.

In [1]:
# Instala as dependências (executar apenas uma vez, se ainda não estiverem instaladas)
#!pip install PyOpenGL PyOpenGL_accelerate glfw PyGLM numpy pillow

In [2]:
import ctypes
import math
import os
from collections import namedtuple
from dataclasses import dataclass

import glfw                    # Janela, contexto OpenGL e eventos
from OpenGL.GL import *        # API OpenGL (pipeline moderno)
import numpy as np             # Vértices e matrizes
import glm                     # Vetores/matrizes da câmera (lookAt, perspective)
from PIL import Image          # Leitura das texturas

from shader_s import Shader    # Classe auxiliar que compila/linka os shaders

### Inicializando a janela

A janela nasce escondida e só é exibida (`glfw.show_window`) logo antes do loop principal.
O tamanho fica em `tela`, que o callback de redimensionamento atualiza.

In [3]:
@dataclass
class Tela:
    #Tamanho atual da janela, em pixels (usado no aspecto da projeção).
    largura: int
    altura: int


TITULO = "Projeto 03 - Casa Minecraft (iluminacao)"
tela = Tela(largura=1000, altura=700)

glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE)

window = glfw.create_window(tela.largura, tela.altura, TITULO, None, None)
if window is None:
    glfw.terminate()
    raise RuntimeError("Failed to create GLFW window")

glfw.make_context_current(window)

### Constrói e compila os shaders

* `vertex_shader.vs`: aplica `projection * view * model` à posição, repassa a coordenada de textura e envia ao
  fragment shader a **posição** e a **normal** do vértice **no mundo** (a normal é multiplicada pela `normalMatrix`).
  Na água, também desliza a textura com o tempo.
* `fragment_shader.fs`: calcula o **modelo de Phong** pixel a pixel:
  * **ambiente**: `ka * luzAmbiente`, igual em qualquer ponto;
  * **difusa**: `kd * max(N·L, 0)`, para cada luz pontual;
  * **especular** (Blinn-Phong): `ks * max(N·H, 0)^brilho`, para cada luz pontual;
  * as luzes pontuais têm **atenuação** com a distância e só contam se estiverem **ligadas** e se forem do
    **mesmo ambiente** (interno/externo) do objeto;
  * `ka`, `kd`, `ks` e `brilho` são o **material de cada objeto** (`material.*` no shader); o teclado ajusta
    fatores globais que multiplicam `kd` e `ks`.

A classe `Shader` guarda o endereço de cada *uniform* na primeira vez que ele é usado, então basta chamar
`shader.setFloat("nome", valor)` (ou `setVec3`, `setInt`, `setBool`, `setMat4`...).

In [4]:
shader = Shader("vertex_shader.vs", "fragment_shader.fs")
shader.use()

shader.setInt("texturaObjeto", 0)    # o sampler do fragment shader lê a textura da unidade 0

## 2. Carregamento: texturas, modelos e geometria

### Geometria da cena

Os vértices, as coordenadas de textura e as **normais** de TODOS os modelos ficam em uma única `Geometria`
(listas paralelas, uma entrada por vértice) e vão para a GPU de uma vez só. Cada modelo guarda apenas
**onde** os seus vértices começam e quantos são.

### Texturas

Todas as texturas são lidas uma única vez (`carrega_textura` usa um cache) e ganham um id próprio com `glGenTextures`.

In [5]:
class Geometria:
    # Todos os vértices da cena (listas paralelas): posição, coordenada de textura e normal.
    def __init__(self):
        self.posicoes = []
        self.coords_textura = []
        self.normais = []

    def __len__(self):
        return len(self.posicoes)

    def adiciona_vertice(self, posicao, coord_textura, normal):
        self.posicoes.append(list(posicao))
        self.coords_textura.append(list(coord_textura))
        self.normais.append(list(normal))


geometria = Geometria()

PASTA_TEXTURAS = 'tex'     # as linhas `usemtl` dos .obj são caminhos relativos a esta pasta (sem extensão)
cache_texturas = {}        # (caminho, wrap) -> id da textura na GPU


def abre_imagem(caminho):
    # Abre a imagem de textura em RGBA. Se não existir, devolve um quadrado magenta e avisa.
    if not os.path.exists(caminho):
        print('Textura nao encontrada:', caminho)
        return Image.new('RGBA', (16, 16), (255, 0, 255, 255))

    imagem = Image.open(caminho).convert('RGBA')
    largura, altura = imagem.size
    if altura > largura and altura % largura == 0:     # textura animada do Minecraft (tira vertical)
        imagem = imagem.crop((0, 0, largura, largura))  # usa só o 1o quadro
    return imagem


def envia_textura_para_gpu(imagem, wrap=GL_REPEAT):
    # Cria uma textura na GPU com os pixels de `imagem` e devolve o seu id.
    texture_id = glGenTextures(1)
    glBindTexture(GL_TEXTURE_2D, texture_id)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_S, wrap)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_T, wrap)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_NEAREST)   # visual "pixelado" do Minecraft
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_NEAREST)
    glPixelStorei(GL_UNPACK_ALIGNMENT, 1)

    pixels = imagem.tobytes("raw", "RGBA", 0, -1)    # -1: inverte o eixo vertical (origem da textura embaixo)
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGBA, imagem.size[0], imagem.size[1], 0, GL_RGBA, GL_UNSIGNED_BYTE, pixels)
    return texture_id


def carrega_textura(caminho, wrap=GL_REPEAT):
    # Carrega (uma única vez) o arquivo de textura `caminho` e devolve o id dela.
    chave = (os.path.normpath(caminho), wrap)
    if chave not in cache_texturas:
        cache_texturas[chave] = envia_textura_para_gpu(abre_imagem(caminho), wrap)
    return cache_texturas[chave]

### Modelos Wavefront (`.obj`)

`le_obj` lê um arquivo **Wavefront**: vértices (`v`), coordenadas de textura (`vt`), **normais (`vn`)**,
faces (`f v/vt/vn`) e a textura de cada grupo de faces (`usemtl`). **Não usamos arquivos `.mtl`**: a própria linha
`usemtl` traz o caminho da imagem, relativo à pasta `tex/` e sem a extensão. Exemplo:

```
usemtl minecraft/block/oak_planks      ->      tex/minecraft/block/oak_planks.png
```

Os modelos de `objetos/` já trazem as **normais** de cada face (unitárias, uma por face: sombreamento *flat*, que combina
com o visual de blocos).

Cada modelo vira um `Modelo`, que é uma lista de **partes** (`Parte`): um intervalo contínuo de vértices
(`inicio`, `qtd`) com **uma textura** e **um ambiente**.
Quando `carrega_modelo` recebe uma função `classifica`, as faces são separadas em partes `'interno'` e `'externo'`
(usado na casa e no terreno). Sem `classifica`, o ambiente da parte fica `None` e vale o do `Objeto`.

In [6]:
@dataclass
class Parte:
    # Um trecho de vértices de um modelo que usa UMA textura e UM ambiente.
    inicio: int               # índice do primeiro vértice em `geometria`
    qtd: int                  # quantidade de vértices
    textura: int              # id da textura na GPU
    material: str = ''        # nome da textura (linha `usemtl`)
    ambiente: str = None      # 'interno' | 'externo' | None (vale o do Objeto)
    transparente: bool = False  # desenhada depois das opacas (água)
    desliza: bool = False       # a textura anda com o tempo (água e lava do lago)
    emissiva: bool = False      # brilha sozinha, sem depender das luzes (olhos do Enderman)


@dataclass
class Modelo:
    nome: str
    partes: list

    def intervalo_de_vertices(self):
        # (primeiro, fim) dos vértices do modelo em `geometria` (as partes são contíguas).
        return self.partes[0].inicio, self.partes[-1].inicio + self.partes[-1].qtd


# Uma face do .obj: a textura dela e os seus vértices, cada um como (i_posicao, i_coord_textura, i_normal)
Face = namedtuple('Face', 'material vertices')


@dataclass
class Malha:
    # Conteúdo cru de um .obj.
    posicoes: list
    coords_textura: list
    normais: list
    faces: list


def le_obj(caminho):
    # Lê um arquivo Wavefront OBJ e devolve uma `Malha` (índices das faces já começando em 0).
    malha = Malha([], [], [], [])
    material = None

    with open(caminho) as arquivo:
        for linha in arquivo:
            valores = linha.split()
            if not valores or valores[0].startswith('#'):
                continue
            tipo, dados = valores[0], valores[1:]

            if tipo == 'v':                  # posição
                malha.posicoes.append([float(x) for x in dados[:3]])
            elif tipo == 'vt':               # coordenada de textura
                malha.coords_textura.append([float(x) for x in dados[:2]])
            elif tipo == 'vn':               # normal
                malha.normais.append([float(x) for x in dados[:3]])
            elif tipo == 'usemtl':           # as próximas faces usam esta textura
                material = dados[0]
            elif tipo == 'f':                # face: cada vértice é "v/vt/vn"
                vertices = [tuple(int(i) - 1 for i in v.split('/')) for v in dados]
                malha.faces.append(Face(material, vertices))

    return malha


def triangula(vertices_da_face):
    # Divide uma face de N vértices em triângulos (leque) e devolve os vértices, de 3 em 3.
    primeiro = vertices_da_face[0]
    triangulos = []
    for i in range(1, len(vertices_da_face) - 1):
        triangulos += [primeiro, vertices_da_face[i], vertices_da_face[i + 1]]
    return triangulos


def carrega_modelo(caminho_obj, classifica=None, ajusta=None):
    #Importa um .obj com as suas texturas e normais e devolve um `Modelo`.

    # classifica: função opcional (face, malha) -> 'interno' | 'externo', usada para separar as faces de
    # um modelo que fica nos dois ambientes (casa e terreno).
    # ajusta: função opcional (malha) -> None que altera a malha logo após a leitura (ex.: girar os braços do Enderman).
    malha = le_obj(caminho_obj)
    if not malha.normais:
        raise ValueError('%s: o modelo nao tem normais (vn); a iluminacao precisa delas' % caminho_obj)

    if ajusta:
        ajusta(malha)

    # Agrupa as faces por (textura, ambiente): cada grupo vira uma Parte
    grupos = {}
    for face in malha.faces:
        if face.material is None:
            raise ValueError('%s: faces antes de qualquer linha usemtl (sem textura)' % caminho_obj)
        ambiente = classifica(face, malha) if classifica else None
        grupos.setdefault((face.material, ambiente), []).append(face)

    nome_modelo = os.path.splitext(os.path.basename(caminho_obj))[0]
    partes = []
    for (material, ambiente), faces in grupos.items():
        inicio = len(geometria)
        for face in faces:
            for i_posicao, i_textura, i_normal in triangula(face.vertices):
                geometria.adiciona_vertice(malha.posicoes[i_posicao],
                                           malha.coords_textura[i_textura],
                                           malha.normais[i_normal])

        nome_textura = os.path.basename(material)
        partes.append(Parte(
            inicio=inicio,
            qtd=len(geometria) - inicio,
            textura=carrega_textura(os.path.join(PASTA_TEXTURAS, material + '.png')),
            material=material,
            ambiente=ambiente,
            transparente='water' in nome_textura,
            desliza=(nome_modelo == 'lago' and 'cobblestone' not in material),   # a pedra do lago fica parada
            emissiva='eyes' in nome_textura,                                      # olhos do Enderman
        ))

    return Modelo(nome_modelo, partes)

### Carregando todos os modelos da cena

Cada item da cena vem de **um `.obj` próprio** (`objetos/`), sem `.mtl`. Os vértices de cada modelo estão em
**coordenadas locais**: a origem é o pivô do item (o centro da sua base), então posicionar ou girar
um item é só montar a sua matriz **Model**.

**Interno x externo na casa e no terreno:** a sala fica no paralelepípedo `CASA_INT_MIN`..`CASA_INT_MAX`
(coordenadas do mundo). Para cada face, a função `classifica` olha um ponto 5 cm **à frente da face**
(`centro_da_face + 0.05 * normal`): se ele cai dentro da sala, a face é **interna** (parede por dentro, piso, forro);
senão é **externa** (parede por fora, telhado, grama). Os demais modelos não precisam disso: o `ambiente` vem do `Objeto`.

**Enderman:** o modelo `enderman.obj` tem os braços caídos; ao carregar, `gira_bracos_enderman` gira os braços para a frente em torno do ombro (`ANGULO_BRACOS`). O `bloco_redstone.obj` é um cubo 1x1x1 que o Enderman carrega entre as mãos (na cena ele é reduzido para 0,5). O terreno inclui os **vidros** (caixa de vidro sobre a caverna de pedra), que ficam em `terreno.obj` com a textura `glass.png`.


In [7]:
PASTA_MODELOS = 'objetos'

NOMES_MODELOS = [
    # estrutura
    'terreno', 'casa', 'porta',
    # dentro da casa
    'cama', 'bau_duplo', 'quadro', 'lanterna', 'vaso_flor', 'tocha_redstone',
    # mesa de trabalho e fornalha aparecem dentro e fora
    'mesa_trabalho', 'fornalha',
    # fora da casa
    'porco', 'cerca', 'lago', 'bau', 'alavanca', 'piston',
    # personagem controlado pelas setas: o Enderman e o bloco de redstone que ele carrega
    'enderman', 'bloco_redstone',
]

# A sala da casa (coordenadas do MUNDO):
# x de 0 a 6, y de 0 (piso) a 4 (forro), z de 2.5 (parede da frente) a 6.5 (parede do fundo)
CASA_INT_MIN = np.array([0.0, 0.0, 2.5])
CASA_INT_MAX = np.array([6.0, 4.0, 6.5])
DISTANCIA_A_FRENTE_DA_FACE = 0.05     # 5 cm


def cria_classificador(pos_modelo):
    # Devolve uma função que diz se uma face do modelo (colocado em `pos_modelo` no mundo) é 'interna' ou 'externa'.
    deslocamento = np.array(pos_modelo)

    def classifica(face, malha):
        pontos = np.array([malha.posicoes[i_posicao] for i_posicao, _, _ in face.vertices])
        # Centro da caixa que envolve a face (e não o centróide do triângulo: em um retângulo dividido em
        # 2 triângulos, o centróide de um deles pode cair exatamente no limite da sala e ser classificado errado)
        centro = (pontos.min(axis=0) + pontos.max(axis=0)) / 2.0
        normal = np.array(malha.normais[face.vertices[0][2]])

        ponto_a_frente = centro + deslocamento + DISTANCIA_A_FRENTE_DA_FACE * normal
        dentro = np.all(ponto_a_frente > CASA_INT_MIN) and np.all(ponto_a_frente < CASA_INT_MAX)
        return 'interno' if dentro else 'externo'

    return classifica


# Só a casa e o terreno (ambos na origem) misturam faces internas e externas
CLASSIFICADORES = {'casa': cria_classificador((0, 0, 0)),
                   'terreno': cria_classificador((0, 0, 0))}

# Enderman: os braços (vértices 25 a 40 do .obj) giram em torno do ombro para a frente (+z), como se segurassem um bloco
ANGULO_BRACOS = 35.0                 # graus (0 = braços caídos)
OMBRO_Y = 2.625                      # altura do ombro (topo dos braços) no modelo
VERTICES_BRACOS = range(24, 40)      # índices (a partir de 0) dos vértices dos dois braços


def gira_bracos_enderman(malha):
    # Gira os vértices e as normais dos braços ANGULO_BRACOS graus em torno do eixo X (que passa pelo ombro).
    c, s = np.cos(np.radians(ANGULO_BRACOS)), np.sin(np.radians(ANGULO_BRACOS))

    for i in VERTICES_BRACOS:
        x, y, z = malha.posicoes[i]
        y -= OMBRO_Y
        malha.posicoes[i] = [x, OMBRO_Y + y * c + z * s, -y * s + z * c]

    # Cada face dos braços ganha uma normal girada (as normais do .obj são compartilhadas com o resto do corpo)
    normal_girada = {}
    for k, face in enumerate(malha.faces):
        if not all(i_posicao in VERTICES_BRACOS for i_posicao, _, _ in face.vertices):
            continue
        novos_vertices = []
        for i_posicao, i_textura, i_normal in face.vertices:
            if i_normal not in normal_girada:
                nx, ny, nz = malha.normais[i_normal]
                malha.normais.append([nx, ny * c + nz * s, -ny * s + nz * c])
                normal_girada[i_normal] = len(malha.normais) - 1
            novos_vertices.append((i_posicao, i_textura, normal_girada[i_normal]))
        malha.faces[k] = Face(face.material, novos_vertices)


AJUSTES = {'enderman': gira_bracos_enderman}

modelos = {}
for nome in NOMES_MODELOS:
    caminho = os.path.join(PASTA_MODELOS, nome + '.obj')
    modelos[nome] = carrega_modelo(caminho, CLASSIFICADORES.get(nome), AJUSTES.get(nome))
    print('%-14s %2d parte(s) com textura' % (nome, len(modelos[nome].partes)))

for nome in CLASSIFICADORES:
    triangulos_por_ambiente = {}
    for parte in modelos[nome].partes:
        triangulos_por_ambiente[parte.ambiente] = triangulos_por_ambiente.get(parte.ambiente, 0) + parte.qtd // 3
    print('%s: triangulos por ambiente' % nome, triangulos_por_ambiente)

terreno         6 parte(s) com textura
casa            4 parte(s) com textura
porta           2 parte(s) com textura
cama            1 parte(s) com textura
bau_duplo       2 parte(s) com textura
quadro          1 parte(s) com textura
lanterna        1 parte(s) com textura
vaso_flor       3 parte(s) com textura
tocha_redstone  1 parte(s) com textura
mesa_trabalho   4 parte(s) com textura
fornalha        3 parte(s) com textura
porco           1 parte(s) com textura
cerca           1 parte(s) com textura
lago            4 parte(s) com textura
bau             1 parte(s) com textura
alavanca        2 parte(s) com textura
piston          3 parte(s) com textura
enderman        2 parte(s) com textura
bloco_redstone  1 parte(s) com textura
casa: triangulos por ambiente {'externo': 662, 'interno': 48}
terreno: triangulos por ambiente {'externo': 452}


### Cubo criado por código: SkyBox (dia e noite)

O **SkyBox** é um cubo de 6 faces com 3 texturas (`lados` nos 4 lados, `topo` e `base`). Existem **dois conjuntos** em `tex/skybox/`:
`lados.png`, `topo.png` e `base.png` (**dia**) e `lados_noite.png`, `topo_noite.png` e `base_noite.png` (**noite**: estrelas, lua e nuvens escuras).
Os lados e a base são só um degradê; as **nuvens e o sol (ou a lua) ficam apenas na face de cima**. O céu **não é iluminado**,
então suas normais (apontando para dentro) só mantêm os buffers alinhados.

A troca é feita por `controles.noite`: com a luz ambiente **≤ 25%** (ou desligada) desenhamos a skybox de noite e com **> 25%** a de dia.

O gerador de faces, `faces_do_cubo`, monta os dois céus.


In [8]:
# Cubo [-1,1]^3. Para cada face: normal (para fora), eixo U e eixo V da textura.
FACES_CUBO = {
    'direita':  ((1, 0, 0),  (0, 0, -1), (0, 1, 0)),
    'esquerda': ((-1, 0, 0), (0, 0, 1),  (0, 1, 0)),
    'topo':     ((0, 1, 0),  (1, 0, 0),  (0, 0, -1)),
    'base':     ((0, -1, 0), (1, 0, 0),  (0, 0, 1)),
    'frente':   ((0, 0, 1),  (1, 0, 0),  (0, 1, 0)),
    'tras':     ((0, 0, -1), (-1, 0, 0), (0, 1, 0)),
}
CANTOS_UV = [(0, 0), (1, 0), (1, 1), (0, 1)]    # cantos de uma face, na ordem em que são percorridos
INDICES_DOS_2_TRIANGULOS = (0, 1, 2, 0, 2, 3)   # uma face (quadrado) = 2 triângulos


def faces_do_cubo(metade):
    # Gera (nome_da_face, normal_para_fora, 6 vértices [(posicao, uv)]) de um cubo de aresta 2*metade centrado na origem.
    for nome, (normal, eixo_u, eixo_v) in FACES_CUBO.items():
        n, U, V = (np.array(x, dtype=np.float32) for x in (normal, eixo_u, eixo_v))
        cantos = [metade * (n + (2*u - 1)*U + (2*v - 1)*V) for (u, v) in CANTOS_UV]
        vertices = [(cantos[i], CANTOS_UV[i]) for i in INDICES_DOS_2_TRIANGULOS]
        yield nome, n, vertices


# SkyBox
LADOS_SKYBOX = ('direita', 'esquerda', 'frente', 'tras')   # os 4 lados usam a mesma imagem (lados.png)


def cria_skybox(sufixo=''):
    # Monta o cubo do céu (uma Parte por face) e carrega as texturas (lados, topo e base).
    # sufixo '' = céu de dia; '_noite' = céu de noite (lados_noite.png, topo_noite.png, base_noite.png).
    partes = []
    for nome, normal_para_fora, vertices in faces_do_cubo(metade=1.0):
        inicio = len(geometria)
        for posicao, uv in vertices:
            geometria.adiciona_vertice(posicao, uv, -normal_para_fora)    # normal para dentro (não é usada)

        arquivo = ('lados' if nome in LADOS_SKYBOX else nome) + sufixo
        # CLAMP_TO_EDGE: evita que a textura "vaze" para o outro lado e crie linhas nas emendas
        textura = carrega_textura(os.path.join(PASTA_TEXTURAS, 'skybox', arquivo + '.png'), wrap=GL_CLAMP_TO_EDGE)
        partes.append(Parte(inicio=inicio, qtd=6, textura=textura, material='skybox%s/%s' % (sufixo, nome)))
    return Modelo('skybox' + sufixo, partes)


skybox_dia = cria_skybox()
skybox_noite = cria_skybox('_noite')

## 3. A cena: instâncias dos modelos, materiais e luzes

Cada elemento de `CENA` é um `Objeto`: **qual modelo usar**, **onde/como colocá-lo** (posição, rotação em Y e escala fixa),
**em qual ambiente ele está** e o seu **material**: `Material(ka, kd, ks, brilho)`, ou seja, as reflexões
**ambiente, difusa e especular** e o **expoente de brilho** *próprios daquele objeto* (a água é bem mais "espelhada" que a
grama, a lanterna de metal brilha mais que a cama...). `emissao` é a luz própria do objeto: lanternas e tochas só
"brilham" enquanto a **luz delas está ligada**; o bloco de redstone do Enderman brilha enquanto a luz dele estiver ligada.

As **luzes pontuais** (`Luz`) ficam presas a um objeto (seguem o objeto se ele se mexer: é o caso do bloco de redstone, que anda com o Enderman), têm
cor, **ambiente que iluminam** e atenuação com a distância `1 / (kc + kl*d + kq*d²)`.
Cada luz tem o seu "interruptor" (`ligada`).

Sistema de coordenadas do mundo: `y = 0` é o chão (topo da grama e do piso de tábuas); `+z` aponta para o fundo da
casa; a unidade é o "bloco" do Minecraft (1 bloco ≈ 1 metro). O quintal fica em `z < 0` e a casa em `z > 0`.

In [9]:
# ka = reflexão ambiente, kd = reflexão difusa, ks = reflexão especular, brilho = expoente especular (quanto maior, mais "polido")
Material = namedtuple('Material', 'ka kd ks brilho')


@dataclass
class Objeto:    
    """
    id:       nome único do objeto.
    modelo:   nome do modelo (chave de `modelos`) que será desenhado.
    pos:      posição (x, y, z) do pivô do modelo no mundo.
    material: Material(ka, kd, ks, brilho) PRÓPRIO deste objeto.
    rot_y:    rotação (graus) em torno do eixo Y (só a orientação fixa na cena).
    escala:   fator de escala fixo (sx, sy, sz) (ex.: o quadro 1x1 vira 2x1).
    ambiente: 'interno', 'externo' ou 'estrutura' (casa/terreno: cada face já foi classificada ao carregar o modelo).
    luz:      id da `Luz` que este objeto representa (ou None). A `emissao` só vale com essa luz ligada.
    emissao:  quanto a textura "brilha sozinha" (0 = nada, 1 = totalmente iluminada por si mesma).
    """
    id: str
    modelo: str
    pos: tuple
    material: Material
    rot_y: float = 0.0
    escala: tuple = (1.0, 1.0, 1.0)
    ambiente: str = 'externo'
    luz: str = None
    emissao: float = 0.0


CENA = [
    # estrutura: piso externo (grama), casa (piso interno de tábuas) e porta
    Objeto('terreno', 'terreno', (0, 0, 0), Material(1.0, 0.90, 0.05, 4),  ambiente='estrutura'),
    Objeto('casa',    'casa',    (0, 0, 0), Material(1.0, 0.85, 0.15, 16), ambiente='estrutura'),
    Objeto('porta',   'porta',   (3, 0, 1), Material(1.0, 0.85, 0.20, 24), ambiente='externo'),   # fica na entrada, do lado de fora

    # ambiente INTERNO (dentro da casa)
    Objeto('cama',         'cama',          (3, 0, 5.5),   Material(1.0, 0.90, 0.08, 6),   ambiente='interno'),
    Objeto('bau_duplo_1',  'bau_duplo',     (0.5, 0, 5.5), Material(1.0, 0.85, 0.30, 32),  ambiente='interno'),
    Objeto('bau_duplo_2',  'bau_duplo',     (5.5, 0, 5.5), Material(1.0, 0.80, 0.45, 48),  rot_y=180, ambiente='interno'),
    Objeto('mesa_int',     'mesa_trabalho', (0.5, 0, 4),   Material(1.0, 0.85, 0.25, 24),  ambiente='interno'),
    Objeto('fornalha_int', 'fornalha',      (5.5, 0, 4),   Material(1.0, 0.90, 0.40, 48),  rot_y=-90, ambiente='interno'),
    # quadro 1x1 com textura 2:1  ->  escala 2 em X; centrado sobre a janela (x de 2 a 4) e a cama
    Objeto('quadro',       'quadro',        (3, 2, 6.47),  Material(1.0, 0.95, 0.60, 64),  escala=(2, 1, 1), ambiente='interno'),
    Objeto('lanterna_1',   'lanterna',      (2.5, 3.062, 5), Material(1.0, 0.80, 0.80, 64), ambiente='interno', luz='lanterna_1', emissao=0.7),
    Objeto('lanterna_2',   'lanterna',      (3.5, 3.062, 5), Material(1.0, 0.80, 0.80, 64), ambiente='interno', luz='lanterna_2', emissao=0.7),
    Objeto('vaso_1',       'vaso_flor',     (2.5, 1, 7),   Material(1.0, 0.90, 0.20, 16),  ambiente='interno'),
    Objeto('vaso_2',       'vaso_flor',     (3.5, 1, 7),   Material(1.0, 0.90, 0.20, 16),  ambiente='interno'),
    # tochas: pouca difusa/especular PRÓPRIA e emissão moderada, para o vermelho não estourar (>1) e a textura aparecer
    Objeto('tocha_1',      'tocha_redstone', (1.5, 1, 2.5), Material(1.0, 0.30, 0.10, 24), ambiente='interno', luz='tocha_1', emissao=0.6),
    Objeto('tocha_2',      'tocha_redstone', (4.5, 1, 2.5), Material(1.0, 0.30, 0.10, 24), ambiente='interno', luz='tocha_2', emissao=0.6),

    # ambiente EXTERNO (quintal)
    Objeto('cerca',        'cerca',         (5.5, 0, -3),  Material(1.0, 0.85, 0.15, 16),  ambiente='externo'),
    Objeto('porco',        'porco',         (5.326, 0, -3.07), Material(1.0, 0.90, 0.20, 12), rot_y=-111.83, ambiente='externo'),
    Objeto('lago',         'lago',          (0, 0, -4),    Material(1.0, 0.75, 0.90, 128), ambiente='externo'),   # água: bem especular
    Objeto('piston',       'piston',        (-0.5, 0, -6), Material(1.0, 0.90, 0.45, 40),  ambiente='externo'),
    Objeto('alavanca',     'alavanca',      (0.5, 0, -6),  Material(1.0, 0.85, 0.35, 32),  ambiente='externo'),
    Objeto('mesa_ext',     'mesa_trabalho', (4.5, 0, -6),  Material(1.0, 0.85, 0.25, 24),  ambiente='externo'),
    Objeto('fornalha_ext', 'fornalha',      (3.5, 0, -6),  Material(1.0, 0.90, 0.40, 48),  ambiente='externo'),
    Objeto('bau',          'bau',           (5.5, 0, -6),  Material(1.0, 0.85, 0.30, 32),  ambiente='externo'),

    # Enderman controlado pelas setas (posição/rotação atualizadas a cada quadro) e o bloco de redstone que ele segura.
    # O bloco .obj (1x1x1) é reduzido para 0.5 (tamanho na mão do Enderman); em X fica um pouco mais fino para não coincidir com a parte interna dos braços (ESCALA_BLOCO).
    Objeto('enderman',       'enderman',       (-5.5, 0, -5.0), Material(1.0, 0.85, 0.10, 8),  ambiente='externo'),
    Objeto('bloco_redstone', 'bloco_redstone', (-5.5, 1.4, -4.2), Material(1.0, 0.85, 0.25, 24), escala=(0.48, 0.5, 0.5), ambiente='externo',
           luz='redstone', emissao=0.7),   # fonte de luz do quintal (a posição é atualizada junto com o Enderman)

]

objeto_por_id = {obj.id: obj for obj in CENA}

AMBIENTES = {'externo': 0, 'interno': 1}      # número que o shader usa para cada ambiente


@dataclass
class Luz:
    """
    id:           nome da luz.
    objeto:       id do objeto onde ela está.
    deslocamento: onde fica a chama/lâmpada em relação ao pivô do objeto.
    cor:          cor (e intensidade) RGB da luz.
    ambiente:     'interno' ou 'externo': a luz SÓ ilumina objetos desse ambiente.
    atenuacao:    (kc, kl, kq) em 1 / (kc + kl*d + kq*d²).
    ligada:       o "interruptor".
    """
    id: str
    objeto: str
    deslocamento: tuple
    cor: tuple
    ambiente: str
    atenuacao: tuple
    ligada: bool = True

    def posicao_no_mundo(self):
        # Posição atual da luz: acompanha o objeto onde ela está.
        pos_objeto = np.array(objeto_por_id[self.objeto].pos, dtype=np.float32)
        return pos_objeto + np.array(self.deslocamento, dtype=np.float32)


ATENUACAO_FRACA = (1.0, 0.14, 0.12)    # lanternas e tochas (alcance curto)

LUZES = [
    # externa: o bloco de redstone que o Enderman carrega (anda com ele pelo quintal)
    Luz('redstone',   'bloco_redstone', (0, 0, 0),   (1.00, 0.20, 0.10), 'externo', (1.0, 0.12, 0.10)),
    # internas:
    # lanternas alaranjadas penduradas no forro
    Luz('lanterna_1', 'lanterna_1', (0, 0.25, 0),    (1.00, 0.55, 0.15), 'interno', ATENUACAO_FRACA),
    Luz('lanterna_2', 'lanterna_2', (0, 0.25, 0),    (1.00, 0.55, 0.15), 'interno', ATENUACAO_FRACA),
    # tochas de redstone vermelhas na parede da frente
    Luz('tocha_1',    'tocha_1',    (0, 0.75, 0.25), (1.00, 0.10, 0.05), 'interno', ATENUACAO_FRACA),
    Luz('tocha_2',    'tocha_2',    (0, 0.75, 0.25), (1.00, 0.10, 0.05), 'interno', ATENUACAO_FRACA),
]

luz_por_id = {luz.id: luz for luz in LUZES}

## 4. Enviando os dados para a GPU

Um VBO para as posições, outro para as coordenadas de textura e um terceiro para as **normais**
(todos os modelos e o céu juntos). `cria_vbo` cria o buffer e liga ele ao atributo (`in`) de mesmo nome no vertex shader.

In [10]:
def cria_vbo(dados, nome_atributo, componentes):
    # Envia `dados` (uma linha por vértice) para a GPU e liga ao atributo `nome_atributo` do vertex shader.
    dados = np.array(dados, dtype=np.float32)

    vbo = glGenBuffers(1)
    glBindBuffer(GL_ARRAY_BUFFER, vbo)
    glBufferData(GL_ARRAY_BUFFER, dados.nbytes, dados, GL_STATIC_DRAW)

    local = shader.attribLocation(nome_atributo)
    glEnableVertexAttribArray(local)
    glVertexAttribPointer(local, componentes, GL_FLOAT, False, dados.strides[0], ctypes.c_void_p(0))
    return vbo


posicoes = np.array(geometria.posicoes, dtype=np.float32)    # (também usado para calcular os limites do cenário)

vbos = [
    cria_vbo(posicoes,                  'posicao',       3),
    cria_vbo(geometria.coords_textura,  'coord_textura', 2),
    cria_vbo(geometria.normais,         'normal',        3),
]

## 5. Matrizes, câmera, controles e teclado

### Matrizes de transformação 3D (Model)

Matrizes 4x4 do numpy (linha a linha), enviadas ao shader com `transpose=True`. A matriz **Model** de cada objeto
é `T * R * S`: primeiro a escala fixa (só o quadro usa: 2x1), depois a rotação (em torno do pivô do modelo,
só para orientar o objeto) e por fim a translação para a posição no mundo.

Para a iluminação, cada objeto também precisa da **matriz normal** (inversa transposta da parte 3x3 da Model), que leva as
normais para o mundo sem deformá-las.

In [11]:
def translacao(tx, ty, tz):
    # Matriz de translação: desloca um ponto em (tx, ty, tz).
    matriz = np.identity(4, dtype=np.float32)
    matriz[:3, 3] = (tx, ty, tz)
    return matriz


def rotacao_y(angulo):
    # Matriz de rotação em torno do eixo Y (`angulo` em radianos).
    c, s = np.cos(angulo), np.sin(angulo)
    return np.array([
         c, 0, s, 0,
         0, 1, 0, 0,
        -s, 0, c, 0,
         0, 0, 0, 1
    ], dtype=np.float32).reshape(4, 4)


def escala(sx, sy, sz):
    # Matriz de escala não-uniforme, com fatores sx, sy e sz em cada eixo.
    return np.diag([sx, sy, sz, 1.0]).astype(np.float32)


def matriz_model(obj):
    # Matriz Model do objeto: T(pos) * Ry(rot) * S(escala).
    return translacao(*obj.pos) @ rotacao_y(np.radians(obj.rot_y)) @ escala(*obj.escala)


def matriz_normal(mat_model):
    # Matriz 3x3 que transforma normais para o mundo: inversa transposta da parte 3x3 da Model.
    return np.linalg.inv(mat_model[:3, :3]).T

### Estado da cena e controles

* **Bloco de redstone:** a luz externa está presa ao bloco que o Enderman carrega, então **anda junto com ele** (setas).
* **Luz ambiente:** `ambiente_ligada` (tecla `1`) e `nivel_ambiente` (`T`/`G`). Com `nivel_ambiente_efetivo <= 25%` (`LIMIAR_NOITE`) é **noite** (`controles.noite`) e o céu troca de skybox.
* **Reflexão difusa e especular:** `fator_difusa` (`Y`/`H`) e `fator_especular` (`U`/`J`) multiplicam o `kd` e o `ks` **de todos os objetos**
  (cada objeto continua com os seus próprios valores).
* **Interruptores:** cada `Luz` tem o seu `ligada` (teclas `2` a `6`).

Tudo isso fica no objeto `controles`. As teclas pressionadas ficam em `controles.teclas`; a cada quadro, `atualiza` usa
o tempo decorrido (`dt`) para que a velocidade seja a mesma em qualquer computador.

In [12]:
# iluminação
NIVEL_AMBIENTE_INICIAL = 0.30      # intensidade inicial da luz ambiente (0 a 1)
NIVEL_AMBIENTE_MAX = 1.0
FATOR_MAX = 3.0                    # limite dos fatores de reflexão difusa/especular
VEL_NIVEL = 0.4                    # variação do nível ambiente por segundo (segurando T/G)
VEL_FATOR = 0.8                    # variação dos fatores por segundo (segurando Y/H/U/J)
BRILHO_CEU_MIN = 0.15              # o céu de dia nunca fica totalmente preto (o de noite já é escuro)
LIMIAR_NOITE = 0.25                # luz ambiente <= 25%: skybox de noite; > 25%: skybox de dia
NIVEL_AMBIENTE_CEU_CLARO = 0.6     # a partir deste nível ambiente o céu já está com brilho total

# teclas 2..6: um interruptor para cada luz pontual
TECLAS_LUZES = {
    glfw.KEY_2: 'redstone',
    glfw.KEY_3: 'lanterna_1',
    glfw.KEY_4: 'lanterna_2',
    glfw.KEY_5: 'tocha_1',
    glfw.KEY_6: 'tocha_2',
}


class Controles:
    # Estado que o usuário altera pelo teclado (fora a câmera e os interruptores das luzes pontuais).
    def __init__(self):
        self.teclas = set()          # teclas atualmente pressionadas
        self.reseta()

    def reseta(self):
        # Volta para as condições iniciais (as teclas pressionadas não mudam).
        self.ambiente_ligada = True                      # tecla 1
        self.nivel_ambiente = NIVEL_AMBIENTE_INICIAL     # T / G
        self.fator_difusa = 1.0                          # Y / H
        self.fator_especular = 1.0                       # U / J
        self.modo_wireframe = False                      # P: malha poligonal

    @property
    def nivel_ambiente_efetivo(self):
        # Nível da luz ambiente enviado ao shader (zero se estiver desligada).
        return self.nivel_ambiente if self.ambiente_ligada else 0.0

    @property
    def noite(self):
        # True quando a luz ambiente está em 25% ou menos (ou desligada): o céu vira a skybox de noite.
        return self.nivel_ambiente_efetivo <= LIMIAR_NOITE


controles = Controles()

### Câmera (matrizes **View** e **Projection**) e limites

* `W A S D` movem a câmera; o mouse direciona; o scroll muda o campo de visão.
* `Camera.limita` mantém a câmera **dentro do terreno** (em X e Z), **acima do piso** e **abaixo do teto do céu**.
  Ela pode atravessar os modelos livremente (a casa, os móveis...), mas não o piso nem o céu.
* No **modo caminhada** (tecla `C`) a câmera fica na altura dos olhos e balança ao andar (*head bobbing*).
* O **SkyBox** é desenhado sempre centrado na câmera, então o céu fica "no infinito" e suas bordas nunca aparecem.
* **view**: `glm.lookAt` a partir da posição/direção da câmera. **projection**: perspectiva.

In [13]:
# limites do cenário
# Caixa do terreno (em x e z), calculada a partir da malha do modelo 'terreno'
terreno = objeto_por_id['terreno']
primeiro, fim = modelos['terreno'].intervalo_de_vertices()
vertices_terreno = posicoes[primeiro:fim]

MARGEM = 0.3                                   # a câmera não encosta exatamente na borda
LIM_X = (vertices_terreno[:, 0].min() + terreno.pos[0] + MARGEM, vertices_terreno[:, 0].max() + terreno.pos[0] - MARGEM)
LIM_Z = (vertices_terreno[:, 2].min() + terreno.pos[2] + MARGEM, vertices_terreno[:, 2].max() + terreno.pos[2] - MARGEM)
ALTURA_MIN = 0.4                               # acima do piso (y = 0)
ALTURA_MAX = 40.0                              # "teto" do céu
print('limites da camera: x', LIM_X, 'z', LIM_Z, 'y', (ALTURA_MIN, ALTURA_MAX))

TAMANHO_CEU = 100.0                            # metade da aresta do cubo do céu (menor que o plano far = 300)

# parâmetros da câmera
POS_INICIAL = (3.0, 6.0, -8.0)                 # no quintal, olhando para a casa
YAW_INICIAL, PITCH_INICIAL = 90.0, -24.0       # yaw 90 = olhando para +z
FOV_INICIAL = 60.0                             # campo de visão vertical (graus)
FOV_MIN, FOV_MAX = 1.0, 90.0
VEL_CAMERA = 6.0                               # blocos por segundo
SENSIBILIDADE_MOUSE = 0.1
ALTURA_DOS_OLHOS = 1.6                         # modo caminhada
BOBBING_VELOCIDADE = 10.0                      # modo caminhada: velocidade do balanço
BOBBING_AMPLITUDE = 0.08                       # modo caminhada: tamanho do balanço
PLANO_PERTO, PLANO_LONGE = 0.1, 300.0


class Camera:
    # Câmera em primeira pessoa: posição, direção (yaw/pitch), zoom e modo de movimento.
    CIMA = glm.vec3(0.0, 1.0, 0.0)

    def __init__(self):
        self.ultimo_mouse = None     # (x, y) do cursor no evento anterior (None antes do 1o movimento)
        self.reseta()

    def reseta(self):
        self.pos = glm.vec3(*POS_INICIAL)
        self.yaw = YAW_INICIAL
        self.pitch = PITCH_INICIAL
        self.fov = FOV_INICIAL
        self.modo_voo = True         # True = voo livre, False = caminhada no chão (head bobbing)
        self.passo_bobbing = 0.0     # contador do balanço da caminhada
        self._atualiza_direcao()

    def _atualiza_direcao(self):
        # Recalcula o vetor `frente` (para onde a câmera olha) a partir de yaw e pitch.
        yaw, pitch = glm.radians(self.yaw), glm.radians(self.pitch)
        self.frente = glm.normalize(glm.vec3(glm.cos(yaw) * glm.cos(pitch),
                                             glm.sin(pitch),
                                             glm.sin(yaw) * glm.cos(pitch)))

    def gira(self, dx, dy):
        # Gira a câmera com o deslocamento do mouse (em pixels).
        self.yaw += dx * SENSIBILIDADE_MOUSE
        self.pitch = min(max(self.pitch + dy * SENSIBILIDADE_MOUSE, -89.0), 89.0)
        self._atualiza_direcao()

    def zoom(self, rolagem):
        # Scroll do mouse: aumenta/diminui o campo de visão.
        self.fov = min(max(self.fov - rolagem, FOV_MIN), FOV_MAX)

    def anda(self, teclas, dt):
        # Move a câmera com W A S D durante `dt` segundos.
        passo = VEL_CAMERA * dt
        direita = glm.normalize(glm.cross(self.frente, self.CIMA))

        andando = False
        for tecla, direcao in ((glfw.KEY_W, self.frente), (glfw.KEY_S, -self.frente),
                               (glfw.KEY_A, -direita),    (glfw.KEY_D, direita)):
            if tecla in teclas:
                self.pos += passo * direcao
                andando = True

        # Modo caminhada: trava a altura nos olhos do jogador e balança ao andar
        if not self.modo_voo:
            self.pos.y = ALTURA_DOS_OLHOS
            if andando:
                self.passo_bobbing += dt * BOBBING_VELOCIDADE
                self.pos.y += math.sin(self.passo_bobbing) * BOBBING_AMPLITUDE

        self._limita()

    def _limita(self):
        # Mantém a câmera dentro do terreno e do céu (os modelos podem ser atravessados).
        self.pos.x = min(max(self.pos.x, LIM_X[0]), LIM_X[1])
        self.pos.z = min(max(self.pos.z, LIM_Z[0]), LIM_Z[1])
        self.pos.y = min(max(self.pos.y, ALTURA_MIN), ALTURA_MAX)

    def matriz_view(self):
        return np.array(glm.lookAt(self.pos, self.pos + self.frente, self.CIMA))

    def matriz_projecao(self, aspecto):
        return np.array(glm.perspective(glm.radians(self.fov), aspecto, PLANO_PERTO, PLANO_LONGE))


camera = Camera()

limites da camera: x (np.float32(-7.7), np.float32(7.7)) z (np.float32(-8.2), np.float32(8.2)) y (0.4, 40.0)


### Enderman controlado pelas setas

* **Direção relativa à câmera:** `↑` anda para onde a câmera olha (projetado no chão), `↓` para trás, `←` e `→` para os lados.
  Várias setas juntas dão diagonais. O Enderman **gira suavemente** (`rot_y`) para a direção em que anda.
* **Bloco de redstone:** acompanha a posição e a rotação do Enderman (fica à frente dele, entre as mãos).
* **Colisão com a casa:** a casa é uma caixa no plano XZ (`CASA_X`, `CASA_Z`, tirada dos vértices da `casa`). O Enderman é
  formado por **dois círculos** (o corpo e o bloco que ele carrega); a cada quadro, qualquer círculo que entre na caixa é
  **empurrado para fora** (então ele desliza pelas paredes e o bloco também não atravessa). A casa inteira, inclusive a porta, é barrada.
* **Tremor:** quando o centro da tela aponta para os olhos do Enderman (`camera_olhando_para_olhos`: dentro de `ANGULO_OLHAR_GRAUS` do centro da tela, até `DIST_MAX_OLHAR` de distância e só se o rosto estiver virado para a câmera), ele treme (`AMPLITUDE_TREMOR`, `FREQUENCIA_TREMOR`). O tremor é só visual: a posição real fica em `pos_base_enderman`, então a colisão e o movimento não são afetados, e o bloco de redstone treme junto.
* **Limites:** o corpo pode ir até as bordas do terreno (o bloco pode passar da borda).


In [14]:
VEL_ENDERMAN = 4.0                 # blocos por segundo
VEL_GIRO_ENDERMAN = 720.0          # graus por segundo (velocidade com que ele vira)
RAIO_CORPO = 0.40                  # círculo de colisão do corpo (os braços chegam a 0.375)
RAIO_BLOCO = 0.35                  # círculo de colisão do bloco (o cubo de 0.5 tem meia-diagonal ~0.35)
MARGEM_ENDERMAN = 0.30             # distância mínima do centro do corpo até a borda do terreno
POS_INICIAL_ENDERMAN = (-5.5, 0.0, -5.0)    # onde o Enderman nasce (no quintal) e para onde volta com o Espaço
DIST_BLOCO_NO_BRACO = 1.45         # onde o bloco fica ao longo do braço, contado a partir do ombro

# onde o bloco fica em relação ao Enderman (no sistema local dele: +z é a frente), na ponta dos braços girados
BLOCO_Y = OMBRO_Y - DIST_BLOCO_NO_BRACO * math.cos(math.radians(ANGULO_BRACOS))
BLOCO_Z = DIST_BLOCO_NO_BRACO * math.sin(math.radians(ANGULO_BRACOS))
ESCALA_BLOCO = (0.48, 0.5, 0.5)    # bloco .obj reduzido para 0.5 (um pouco mais fino em X para não coincidir com os braços); igual à escala do `Objeto`

enderman = objeto_por_id['enderman']
bloco_redstone = objeto_por_id['bloco_redstone']

# Obstáculo: a casa, vista de cima (menor retângulo em X e Z que contém as partes acima do chão)
casa = objeto_por_id['casa']
primeiro, fim = modelos['casa'].intervalo_de_vertices()
vertices_casa = posicoes[primeiro:fim]
vertices_casa = vertices_casa[vertices_casa[:, 1] >= 0.0]
CASA_X = (vertices_casa[:, 0].min() + casa.pos[0], vertices_casa[:, 0].max() + casa.pos[0])
CASA_Z = (vertices_casa[:, 2].min() + casa.pos[2], vertices_casa[:, 2].max() + casa.pos[2])

# Limites do corpo: todo o quadrado do terreno
ENDER_X = (LIM_X[0] - MARGEM + MARGEM_ENDERMAN, LIM_X[1] + MARGEM - MARGEM_ENDERMAN)
ENDER_Z = (LIM_Z[0] - MARGEM + MARGEM_ENDERMAN, LIM_Z[1] + MARGEM - MARGEM_ENDERMAN)
print('casa (obstaculo): x', CASA_X, 'z', CASA_Z)
print('enderman pode andar em: x', ENDER_X, 'z', ENDER_Z)

# (tecla, quanto anda para a frente da câmera, quanto anda para a direita da câmera)
TECLAS_SETAS = ((glfw.KEY_UP, 1, 0), (glfw.KEY_DOWN, -1, 0), (glfw.KEY_RIGHT, 0, 1), (glfw.KEY_LEFT, 0, -1))

# Tremor: o Enderman treme enquanto a câmera olha para os olhos dele
ALTURA_OLHOS_ENDERMAN = 2.875      # altura do centro da face dos olhos no .obj (a cabeça vai de 2.625 a 3.125)
DESLOC_Z_OLHOS = 0.252             # posição Z (local) da face dos olhos, na frente da cabeça
ANGULO_OLHAR_GRAUS = 8.0           # quanto o olhar pode sair do centro da tela e ainda contar como "olhando"
DIST_MAX_OLHAR = 25.0              # distância máxima (em blocos) da câmera até os olhos para ele tremer
COS_LIMIAR_OLHAR = math.cos(math.radians(ANGULO_OLHAR_GRAUS))    # o teste usa o cosseno do ângulo (sem precisar de acos)
AMPLITUDE_TREMOR = 0.04            # deslocamento máximo do tremor, em blocos
FREQUENCIA_TREMOR = 70.0           # velocidade da oscilação (radianos por segundo)

# Posição real (x, z) do Enderman, sem o tremor. `enderman.pos` recebe esta posição somada ao tremor, então o movimento
# e a colisão partem sempre daqui (o tremor não se acumula nem empurra o Enderman).
pos_base_enderman = [POS_INICIAL_ENDERMAN[0], POS_INICIAL_ENDERMAN[2]]


def camera_olhando_para_olhos(ex, ez, rot_y):
    # True se o centro da tela aponta para os olhos do Enderman (em (ex, ez), virado para rot_y).
    # Direção para onde o Enderman está olhando (+z local rotacionado por rot_y)
    a = math.radians(rot_y)
    frente_ex, frente_ez = math.sin(a), math.cos(a)

    # Posição exata do centro dos olhos no mundo
    ox = ex + frente_ex * DESLOC_Z_OLHOS
    oy = ALTURA_OLHOS_ENDERMAN
    oz = ez + frente_ez * DESLOC_Z_OLHOS

    # Vetor da câmera até os olhos
    dx = ox - camera.pos.x
    dy = oy - camera.pos.y
    dz = oz - camera.pos.z
    dist = math.sqrt(dx * dx + dy * dy + dz * dz)
    if dist < 1e-6 or dist > DIST_MAX_OLHAR:
        return False

    # Só ativa se a câmera estiver na frente do rosto (evita disparar olhando para a nuca)
    if (-dx) * frente_ex + (-dz) * frente_ez <= 0.0:
        return False

    # Direção para onde a câmera está apontando
    yaw = math.radians(camera.yaw)
    pitch = math.radians(camera.pitch)
    fx = math.cos(pitch) * math.cos(yaw)
    fy = math.sin(pitch)
    fz = math.cos(pitch) * math.sin(yaw)

    cos_angulo = (dx * fx + dy * fy + dz * fz) / dist
    return cos_angulo >= COS_LIMIAR_OLHAR


def reseta_enderman():
    # Volta o Enderman e o bloco para a posição, rotação e escala iniciais (sem tremor).
    pos_base_enderman[0] = POS_INICIAL_ENDERMAN[0]
    pos_base_enderman[1] = POS_INICIAL_ENDERMAN[2]
    enderman.modelo = 'enderman'
    enderman.pos = POS_INICIAL_ENDERMAN
    enderman.rot_y = 0.0
    bloco_redstone.modelo = 'bloco_redstone'
    bloco_redstone.escala = ESCALA_BLOCO
    posiciona_bloco()


def direcao_desejada(teclas):
    # Direção (dx, dz) unitária no mundo pedida pelas setas, relativa à câmera; None se nenhuma seta está apertada.
    yaw = math.radians(camera.yaw)
    frente = (math.cos(yaw), math.sin(yaw))          # para onde a câmera olha, no chão
    direita = (-frente[1], frente[0])                # à direita da câmera (mesma conta de Camera.anda)
    dx = dz = 0.0
    for tecla, f, d in TECLAS_SETAS:
        if tecla in teclas:
            dx += f * frente[0] + d * direita[0]
            dz += f * frente[1] + d * direita[1]
    tamanho = math.hypot(dx, dz)
    return None if tamanho < 1e-6 else (dx / tamanho, dz / tamanho)


def gira_para(atual, alvo, passo_max):
    # Gira `atual` (graus) em direção a `alvo` pelo caminho mais curto, no máximo `passo_max` graus.
    diferenca = (alvo - atual + 180.0) % 360.0 - 180.0
    return (atual + min(max(diferenca, -passo_max), passo_max)) % 360.0


def circulos_enderman(x, z, rot_y):
    # Círculos de colisão (x, z, raio) do corpo e do bloco, para o Enderman em (x, z) olhando para rot_y.
    a = math.radians(rot_y)
    return [(x, z, RAIO_CORPO),
            (x + math.sin(a) * BLOCO_Z, z + math.cos(a) * BLOCO_Z, RAIO_BLOCO)]


def empurra_para_fora_da_casa(x, z, rot_y):
    # Tira da caixa da casa qualquer círculo do Enderman que tenha entrado nela (desliza pelas paredes).
    for _ in range(4):
        for cx, cz, raio in circulos_enderman(x, z, rot_y):
            px = min(max(cx, CASA_X[0]), CASA_X[1])      # ponto da caixa mais perto do centro do círculo
            pz = min(max(cz, CASA_Z[0]), CASA_Z[1])
            dx, dz = cx - px, cz - pz
            dist = math.hypot(dx, dz)
            if dist >= raio:
                continue                                  # não encosta na casa
            if dist > 1e-6:
                nx, nz, empurrao = dx / dist, dz / dist, raio - dist
            else:                                         # o centro está dentro da caixa: sai pelo lado mais perto
                lados = [(cx - CASA_X[0], -1, 0), (CASA_X[1] - cx, 1, 0), (cz - CASA_Z[0], 0, -1), (CASA_Z[1] - cz, 0, 1)]
                d, nx, nz = min(lados)
                empurrao = d + raio
            x += nx * empurrao
            z += nz * empurrao
    return x, z


def posiciona_bloco():
    # O bloco de redstone fica à frente do Enderman, entre as mãos, e gira junto com ele (e treme junto, pois parte de `enderman.pos`).
    x, _, z = enderman.pos
    a = math.radians(enderman.rot_y)
    bloco_redstone.pos = (x + math.sin(a) * BLOCO_Z, BLOCO_Y, z + math.cos(a) * BLOCO_Z)
    bloco_redstone.rot_y = enderman.rot_y


def move_enderman(teclas, dt):
    # Anda e vira o Enderman com as setas (relativas à câmera), respeitando o terreno e a casa.
    x, z = pos_base_enderman          # parte da posição sem tremor
    rot_y = enderman.rot_y

    direcao = direcao_desejada(teclas)
    if direcao is not None:
        x += direcao[0] * VEL_ENDERMAN * dt
        z += direcao[1] * VEL_ENDERMAN * dt
        alvo = math.degrees(math.atan2(direcao[0], direcao[1]))     # rot_y que deixa a frente (+z) na direção do passo
        rot_y = gira_para(rot_y, alvo, VEL_GIRO_ENDERMAN * dt)

    for _ in range(2):    # empurra para fora da casa e volta para dentro do terreno
        x, z = empurra_para_fora_da_casa(x, z, rot_y)
        x = min(max(x, ENDER_X[0]), ENDER_X[1])
        z = min(max(z, ENDER_Z[0]), ENDER_Z[1])

    pos_base_enderman[0], pos_base_enderman[1] = x, z

    # Tremor visual (só na posição desenhada): oscila em X e Z enquanto a câmera olha para os olhos
    tx = tz = 0.0
    if camera_olhando_para_olhos(x, z, rot_y):
        t = glfw.get_time() * FREQUENCIA_TREMOR
        tx = AMPLITUDE_TREMOR * math.sin(t)
        tz = AMPLITUDE_TREMOR * math.cos(t * 1.3)

    enderman.pos = (x + tx, 0.0, z + tz)
    enderman.rot_y = rot_y
    posiciona_bloco()


reseta_enderman()

casa (obstaculo): x (np.float32(-1.0), np.float32(7.0)) z (np.float32(0.5), np.float32(8.5))
enderman pode andar em: x (np.float32(-7.7), np.float32(7.7)) z (np.float32(-8.2), np.float32(8.2))


### Teclado e mouse

`key_event` trata os eventos **pontuais** de teclado: os interruptores das luzes (`1` a `6`), `P`, `C`, Espaço e ESC.
Os ajustes **contínuos** (segurar `T G Y H U J`) são tratados em `atualiza`.

In [15]:
def reseta_cena():
    # Volta tudo para as condições iniciais (Espaço).
    controles.reseta()                  # luz ambiente, fatores difuso/especular e modo malha
    camera.reseta()
    reseta_enderman()                   # posição, rotação e tremor do Enderman (e o bloco)
    for luz in LUZES:
        luz.ligada = True


def key_event(window, key, scancode, action, mods):
    if action == glfw.RELEASE:
        controles.teclas.discard(key)
        return
    if action != glfw.PRESS:
        return

    controles.teclas.add(key)

    if key == glfw.KEY_ESCAPE:
        glfw.set_window_should_close(window, True)
    elif key == glfw.KEY_P:                      # sólido <-> malha poligonal
        controles.modo_wireframe = not controles.modo_wireframe
    elif key == glfw.KEY_C:                      # voo <-> caminhada
        camera.modo_voo = not camera.modo_voo
    elif key == glfw.KEY_1:                      # luz ambiente
        controles.ambiente_ligada = not controles.ambiente_ligada
    elif key in TECLAS_LUZES:                    # 2 a 6: bloco de redstone, lanternas, tochas
        luz = luz_por_id[TECLAS_LUZES[key]]
        luz.ligada = not luz.ligada
    elif key == glfw.KEY_SPACE:
        reseta_cena()


def framebuffer_size_callback(window, largura, altura):
    tela.largura, tela.altura = max(largura, 1), max(altura, 1)
    glViewport(0, 0, largura, altura)


def mouse_callback(window, x, y):
    if camera.ultimo_mouse is not None:
        dx = x - camera.ultimo_mouse[0]
        dy = camera.ultimo_mouse[1] - y      # invertido: as coordenadas y da janela crescem para baixo
        camera.gira(dx, dy)
    camera.ultimo_mouse = (x, y)


def scroll_callback(window, xoffset, yoffset):
    camera.zoom(yoffset)


glfw.set_key_callback(window, key_event)
glfw.set_framebuffer_size_callback(window, framebuffer_size_callback)
glfw.set_cursor_pos_callback(window, mouse_callback)
glfw.set_scroll_callback(window, scroll_callback)

glfw.set_input_mode(window, glfw.CURSOR, glfw.CURSOR_DISABLED)

## 6. Atualização e desenho da cena

* `atualiza(dt)`: move a câmera, **anda com o Enderman** (e a luz do bloco de redstone junto; é aqui que ele treme quando olhado) e aplica os ajustes contínuos de luz ambiente, difusa e especular.
* `envia_iluminacao()`: manda para o shader a luz ambiente, os fatores difuso/especular e o vetor de luzes
  (posição atual, cor, ambiente, interruptor e atenuação).
* `desenha_partes` / `desenha_objeto`: antes de desenhar cada objeto, enviam o **material dele** (`ka`, `kd`, `ks`, `brilho`),
  a emissão e o **ambiente** (do objeto ou, na casa e no terreno, da parte).
* `desenha_cena()`: primeiro o **céu** (de dia ou de noite, conforme `controles.noite`; sem iluminação, sempre preenchido, sem escrever profundidade), depois os objetos
  (peças opacas e, por último, as transparentes: a água).

In [16]:
def ajusta_com_teclas(valor, tecla_mais, tecla_menos, velocidade, dt, minimo, maximo):
    # Aumenta `valor` enquanto `tecla_mais` está pressionada e diminui com `tecla_menos` (limitado a [minimo, maximo]).
    if tecla_mais in controles.teclas:
        valor += velocidade * dt
    if tecla_menos in controles.teclas:
        valor -= velocidade * dt
    return min(max(valor, minimo), maximo)


def atualiza(dt):
    # Avança a câmera, o Enderman e os controles de iluminação em `dt` segundos.
    camera.anda(controles.teclas, dt)
    move_enderman(controles.teclas, dt)     # setas: Enderman (e o bloco que ele carrega)

    # Ajustes contínuos: T/G (ambiente), Y/H (difusa), U/J (especular)
    controles.nivel_ambiente = ajusta_com_teclas(controles.nivel_ambiente, glfw.KEY_T, glfw.KEY_G,
                                                 VEL_NIVEL, dt, 0.0, NIVEL_AMBIENTE_MAX)
    controles.fator_difusa = ajusta_com_teclas(controles.fator_difusa, glfw.KEY_Y, glfw.KEY_H,
                                               VEL_FATOR, dt, 0.0, FATOR_MAX)
    controles.fator_especular = ajusta_com_teclas(controles.fator_especular, glfw.KEY_U, glfw.KEY_J,
                                                  VEL_FATOR, dt, 0.0, FATOR_MAX)


def envia_iluminacao():
    # Envia ao shader tudo que é global na iluminação: ambiente, fatores, câmera e as luzes pontuais.
    # Luz ambiente (branca) e céu, que escurece/clareia junto com ela (dia/noite)
    nivel = controles.nivel_ambiente_efetivo
    shader.setVec3('luzAmbiente', nivel, nivel, nivel)
    if controles.noite:
        brilho_ceu = 1.0       # a textura de noite já é escura
    else:
        claridade_ceu = min(nivel / NIVEL_AMBIENTE_CEU_CLARO, 1.0)
        brilho_ceu = BRILHO_CEU_MIN + (1.0 - BRILHO_CEU_MIN) * claridade_ceu
    shader.setFloat('brilhoCeu', brilho_ceu)

    shader.setFloat('fatorDifusa', controles.fator_difusa)
    shader.setFloat('fatorEspecular', controles.fator_especular)
    shader.setVec3('posCamera', camera.pos.x, camera.pos.y, camera.pos.z)

    # Luzes pontuais: a posição acompanha o objeto onde a luz está
    shader.setInt('numLuzes', len(LUZES))
    for i, luz in enumerate(LUZES):
        campo = 'luzes[%d].' % i
        shader.setVec3(campo + 'pos', *luz.posicao_no_mundo())
        shader.setVec3(campo + 'cor', *luz.cor)
        shader.setInt(campo + 'ambiente', AMBIENTES[luz.ambiente])
        shader.setBool(campo + 'ligada', luz.ligada)
        shader.setFloat(campo + 'kc', luz.atenuacao[0])
        shader.setFloat(campo + 'kl', luz.atenuacao[1])
        shader.setFloat(campo + 'kq', luz.atenuacao[2])


def desenha_partes(modelo, ambiente=None, transparente=None, emissao=0.0):
    #Desenha as partes de um modelo.
    """
    ambiente:     o do objeto (a parte pode ter o seu próprio, como na casa e no terreno).
    transparente: True/False desenha só as partes transparentes/opacas; None desenha todas.
    emissao:      luz própria do objeto (as partes `emissiva` brilham por inteiro, como os olhos do Enderman).
    """
    for parte in modelo.partes:
        if transparente is not None and parte.transparente != transparente:
            continue

        shader.setFloat('material.emissao', 1.0 if parte.emissiva else emissao)
        shader.setBool('ehAgua', parte.desliza)
        shader.setInt('ambienteObjeto', AMBIENTES.get(parte.ambiente or ambiente, 0))
        glBindTexture(GL_TEXTURE_2D, parte.textura)
        glDrawArrays(GL_TRIANGLES, parte.inicio, parte.qtd)


def desenha_objeto(obj, mat_model, mat_normal, transparente):
    # Envia a matriz Model, a matriz normal e o MATERIAL do objeto e desenha as suas partes.
    shader.setMat4('model', mat_model)
    shader.setMat3('normalMatrix', mat_normal)

    shader.setFloat('material.ka', obj.material.ka)
    shader.setFloat('material.kd', obj.material.kd)
    shader.setFloat('material.ks', obj.material.ks)
    shader.setFloat('material.brilho', obj.material.brilho)

    # Lanternas, tochas e o bloco de redstone só brilham com a luz deles ligada
    brilha = obj.luz is None or luz_por_id[obj.luz].ligada
    desenha_partes(modelos[obj.modelo], obj.ambiente, transparente, obj.emissao if brilha else 0.0)


def desenha_ceu():
    # SkyBox: sem iluminação, sempre preenchido, centrado na câmera e sem escrever profundidade.
    glPolygonMode(GL_FRONT_AND_BACK, GL_FILL)
    glDepthMask(GL_FALSE)
    shader.setBool('ehCeu', True)

    mat_ceu = translacao(camera.pos.x, camera.pos.y, camera.pos.z) @ escala(TAMANHO_CEU, TAMANHO_CEU, TAMANHO_CEU)
    shader.setMat4('model', mat_ceu)
    shader.setMat3('normalMatrix', np.identity(3))
    desenha_partes(skybox_noite if controles.noite else skybox_dia)     # <= 25% de luz ambiente: noite

    shader.setBool('ehCeu', False)
    glDepthMask(GL_TRUE)


def desenha_objetos():
    # Objetos iluminados da cena: primeiro os opacos e, por último, os transparentes (água).
    glPolygonMode(GL_FRONT_AND_BACK, GL_LINE if controles.modo_wireframe else GL_FILL)

    matrizes = [matriz_model(obj) for obj in CENA]
    matrizes_normais = [matriz_normal(m) for m in matrizes]
    for transparente in (False, True):
        for obj, mat_model, mat_normal in zip(CENA, matrizes, matrizes_normais):
            desenha_objeto(obj, mat_model, mat_normal, transparente)


def desenha_cena():
    shader.setFloat('tempo', glfw.get_time())     # para a animação da água
    envia_iluminacao()
    desenha_ceu()
    desenha_objetos()


def atualiza_titulo():
    # Mostra o estado da iluminação no título da janela.
    onoff = lambda ligada: 'ON' if ligada else 'off'
    estado = lambda id_luz: onoff(luz_por_id[id_luz].ligada)
    glfw.set_window_title(window, '%s | amb %s %d%% | difusa x%.2f | espec x%.2f | redstone %s | lant %s %s | tocha %s %s | ceu %s' % (
        TITULO, onoff(controles.ambiente_ligada), round(controles.nivel_ambiente * 100),
        controles.fator_difusa, controles.fator_especular,
        estado('redstone'), estado('lanterna_1'), estado('lanterna_2'), estado('tocha_1'), estado('tocha_2'),
        'noite' if controles.noite else 'dia'))

### Nesse momento, nós exibimos a janela!

In [17]:
glfw.show_window(window)

### Loop principal da janela

In [ ]:
glEnable(GL_DEPTH_TEST)    # importante para 3D
glEnable(GL_BLEND)         # transparência (água)
glBlendFunc(GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA)

COR_FUNDO = (0.53, 0.81, 0.92, 1.0)
DT_MAX = 0.1               # limita o passo de tempo (ex.: janela arrastada)
INTERVALO_TITULO = 0.2     # o título da janela é atualizado 5 vezes por segundo

instante_anterior = glfw.get_time()
instante_titulo = 0.0

while not glfw.window_should_close(window):
    agora = glfw.get_time()
    dt = min(agora - instante_anterior, DT_MAX)
    instante_anterior = agora

    glfw.poll_events()
    atualiza(dt)

    if agora - instante_titulo > INTERVALO_TITULO:
        atualiza_titulo()
        instante_titulo = agora

    glClearColor(*COR_FUNDO)
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)

    shader.setMat4('view', camera.matriz_view())
    shader.setMat4('projection', camera.matriz_projecao(tela.largura / tela.altura))

    desenha_cena()

    glfw.swap_buffers(window)

glfw.terminate()